

Required resources in the same Kaggle input dataset:

```text
train.npz
robust_resnet18_exp1.pt
```


In [ ]:
from pathlib import Path
import torch

RUN_FULL_TRAINING = False  # Set True to reproduce the checkpoint from training stages.
ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
DATA_PATH = next(ROOT.rglob('train.npz'))
FINAL_CHECKPOINT = next(ROOT.rglob('robust_resnet18_exp2.pt')) if not RUN_FULL_TRAINING else Path('/kaggle/working/robust_resnet18_exp2.pt')
print('DATA_PATH =', DATA_PATH)
print('FINAL_CHECKPOINT =', FINAL_CHECKPOINT)
print('CUDA =', torch.cuda.is_available())


In [ ]:

from __future__ import annotations

import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, resnet34, resnet50

NUM_CLASSES = 9
MODEL_FACTORIES = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device():
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        assert major >= 7, "This PyTorch build does not support P100; use T4/A100/L4."
        return torch.device("cuda")
    return torch.device("cpu")


def build_model(name="resnet18"):
    model = MODEL_FACTORIES[name](weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


class ImageDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index].float().div(255.0)
        label = self.labels[index]
        if self.augment:
            image = F.pad(image, (4, 4, 4, 4), mode="reflect")
            top = torch.randint(0, 9, ()).item()
            left = torch.randint(0, 9, ()).item()
            image = image[:, top:top + 32, left:left + 32]
            if torch.rand(()) < 0.5:
                image = image.flip(-1)
        return image, label


def stratified_split(labels, validation_fraction=0.1, seed=42):
    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    for label in np.unique(labels):
        idx = np.flatnonzero(labels == label)
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * validation_fraction)))
        val_parts.append(idx[:n_val])
        train_parts.append(idx[n_val:])
    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def load_datasets(path, validation_fraction=0.1, seed=42):
    with np.load(path) as archive:
        images = archive["images"]
        labels = archive["labels"]
    train_idx, val_idx = stratified_split(labels, validation_fraction, seed)
    return ImageDataset(images[train_idx], labels[train_idx], True), ImageDataset(images[val_idx], labels[val_idx], False)


def pgd_linf(model, images, labels, epsilon=8/255, step_size=2/255, steps=7, random_start=True):
    was_training = model.training
    model.eval()
    clean = images.detach()
    if random_start:
        adv = (clean + torch.empty_like(clean).uniform_(-epsilon, epsilon)).clamp(0, 1)
    else:
        adv = clean.clone()
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = F.cross_entropy(model(adv), labels)
        grad = torch.autograd.grad(loss, adv)[0]
        adv = adv.detach() + step_size * grad.sign()
        adv = (clean + (adv - clean).clamp(-epsilon, epsilon)).clamp(0, 1).detach()
    model.train(was_training)
    return adv


def trades_linf(model, images, epsilon=8/255, step_size=2/255, steps=10):
    was_training = model.training
    model.eval()
    clean = images.detach()
    with torch.no_grad():
        clean_probs = F.softmax(model(clean), dim=1).clamp_min(1e-8)
    adv = (clean + torch.empty_like(clean).uniform_(-epsilon, epsilon)).clamp(0, 1)
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = F.kl_div(F.log_softmax(model(adv), dim=1), clean_probs, reduction="batchmean")
        grad = torch.autograd.grad(loss, adv)[0]
        adv = adv.detach() + step_size * grad.sign()
        adv = (clean + (adv - clean).clamp(-epsilon, epsilon)).clamp(0, 1).detach()
    model.train(was_training)
    return adv


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for avg, cur in zip(self.model.state_dict().values(), model.state_dict().values()):
            if avg.dtype.is_floating_point:
                avg.mul_(self.decay).add_(cur, alpha=1 - self.decay)
            else:
                avg.copy_(cur)


class AWP:
    def __init__(self, model, gamma=0.005, lr=0.01):
        self.proxy = copy.deepcopy(model)
        self.opt = torch.optim.SGD(self.proxy.parameters(), lr=lr)
        self.gamma = gamma
        self.backup = {}

    def calc(self, model, images, labels, clean_weight=0.30):
        self.proxy.load_state_dict(model.state_dict())
        self.proxy.train()
        self.opt.zero_grad(set_to_none=True)
        adv = pgd_linf(self.proxy, images, labels, steps=3)
        clean_logits = self.proxy(images)
        adv_logits = self.proxy(adv)
        clean_loss = F.cross_entropy(clean_logits, labels)
        adv_loss = F.cross_entropy(adv_logits, labels)
        loss = clean_weight * clean_loss + (1 - clean_weight) * adv_loss
        (-loss).backward()
        self.opt.step()
        perturb = {}
        model_state = model.state_dict()
        proxy_state = self.proxy.state_dict()
        for name, param in model.named_parameters():
            if param.requires_grad and param.ndim > 1:
                diff = proxy_state[name] - model_state[name]
                dn, wn = diff.norm(), model_state[name].norm()
                if torch.isfinite(dn) and dn > 0 and wn > 0:
                    perturb[name] = self.gamma * wn / dn * diff
        return perturb

    @torch.no_grad()
    def perturb(self, model, perturb):
        self.backup = {}
        for name, param in model.named_parameters():
            if name in perturb:
                self.backup[name] = param.data.clone()
                param.add_(perturb[name])

    @torch.no_grad()
    def restore(self, model):
        for name, param in model.named_parameters():
            if name in self.backup:
                param.copy_(self.backup[name])
        self.backup = {}


def lr_for_epoch(base_lr, epoch, epochs, warmup_epochs):
    if epoch <= warmup_epochs:
        return base_lr * epoch / max(1, warmup_epochs)
    if epoch > int(0.9 * epochs):
        return base_lr * 0.01
    if epoch > int(0.75 * epochs):
        return base_lr * 0.1
    return base_lr


def accuracy(logits, labels):
    return int(logits.argmax(1).eq(labels).sum().item())


def evaluate_clean_pgd(model, loader, device, attack_steps=20, max_batches=None):
    model.eval()
    clean_correct = robust_correct = total = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            clean_correct += accuracy(model(images), labels)
        adv = pgd_linf(model, images, labels, steps=attack_steps)
        with torch.no_grad():
            robust_correct += accuracy(model(adv), labels)
        total += labels.numel()
    return clean_correct / total, robust_correct / total


def evaluate_fgsm_pgd(checkpoint_path, data_path, model_name="resnet18", batch_size=128, workers=4):
    device = choose_device()
    _, val_ds = load_datasets(data_path)
    loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=device.type == "cuda")
    model = build_model(model_name)
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert not any(k.startswith("module.") for k in state), "DataParallel prefix found"
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    clean, fgsm = evaluate_clean_pgd(model, loader, device, attack_steps=1)
    _, pgd20 = evaluate_clean_pgd(model, loader, device, attack_steps=20)
    print(f"clean_accuracy={clean:.4f}")
    print(f"fgsm_accuracy={fgsm:.4f}")
    print(f"pgd20_accuracy={pgd20:.4f}")
    print(f"clean_pgd_unified_score={0.5 * (clean + pgd20):.4f}")


def verify_submission(checkpoint_path, model_name="resnet18"):
    model = build_model(model_name).eval()
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert not any(k.startswith("module.") for k in state), "DataParallel prefix found"
    model.load_state_dict(state, strict=True)
    with torch.no_grad():
        out = model(torch.randn(1, 3, 32, 32))
    assert tuple(out.shape) == (1, 9), tuple(out.shape)
    print(f"OK: {checkpoint_path} loads as {model_name} and outputs (1, 9)")


def train_stage(
    data_path,
    output_path,
    init_checkpoint=None,
    model_name="resnet18",
    objective="mixed",
    trades_beta=6.0,
    clean_weight=0.25,
    epochs=1,
    batch_size=128,
    learning_rate=0.005,
    warmup_epochs=1,
    ema_decay=0.99,
    grad_clip=1.0,
    attack_steps=10,
    validation_attack_steps=20,
    max_validation_batches=16,
    awp_gamma=0.0,
    awp_lr=0.01,
    workers=4,
):
    set_seed(42)
    device = choose_device()
    train_ds, val_ds = load_datasets(data_path)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=device.type == "cuda")
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=device.type == "cuda")
    model = build_model(model_name)
    if init_checkpoint is not None:
        model.load_state_dict(torch.load(init_checkpoint, map_location="cpu", weights_only=True), strict=True)
        print("initialized from", init_checkpoint)
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    ema = EMA(model, ema_decay)
    awp = AWP(model, gamma=awp_gamma, lr=awp_lr) if awp_gamma > 0 else None
    best = -1
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, epochs + 1):
        started = time.time()
        lr = lr_for_epoch(learning_rate, epoch, epochs, warmup_epochs)
        for group in opt.param_groups:
            group["lr"] = lr
        model.train()
        loss_sum = seen = skipped = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if objective == "trades":
                adv = trades_linf(model, images, steps=attack_steps)
            else:
                adv = pgd_linf(model, images, labels, steps=attack_steps)
            opt.zero_grad(set_to_none=True)
            if awp is not None:
                perturb = awp.calc(model, images, labels, clean_weight)
                awp.perturb(model, perturb)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                clean_logits = model(images)
                adv_logits = model(adv)
                clean_loss = F.cross_entropy(clean_logits, labels)
                if objective == "trades":
                    robust_loss = F.kl_div(F.log_softmax(adv_logits, dim=1), F.softmax(clean_logits, dim=1).clamp_min(1e-8), reduction="batchmean")
                    loss = clean_loss + trades_beta * robust_loss
                else:
                    adv_loss = F.cross_entropy(adv_logits, labels)
                    loss = clean_weight * clean_loss + (1 - clean_weight) * adv_loss
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            if awp is not None:
                awp.restore(model)
            scaler.step(opt)
            scaler.update()
            ema.update(model)
            loss_sum += loss.item() * labels.numel()
            seen += labels.numel()
        clean, robust = evaluate_clean_pgd(ema.model, val_loader, device, validation_attack_steps, max_validation_batches)
        score = 0.5 * (clean + robust)
        if score > best:
            best = score
            torch.save(ema.model.state_dict(), output_path)
        print(json.dumps({"epoch": epoch, "learning_rate": lr, "objective": objective, "loss": loss_sum / max(seen, 1), "skipped_batches": skipped, "clean_accuracy": clean, "robust_accuracy": robust, "unified_score": score, "best_score": best, "seconds": round(time.time() - started, 1)}))
    print("saved best state dict to", output_path)
    return output_path


In [ ]:

if RUN_FULL_TRAINING:
    BASE = Path('/kaggle/working/robust_resnet18_exp1.pt')
    FINAL_CHECKPOINT = Path('/kaggle/working/robust_resnet18_exp2.pt')

    train_stage(
        DATA_PATH, BASE,
        objective='trades', trades_beta=6.0,
        epochs=100, batch_size=128, learning_rate=0.05,
        warmup_epochs=5, ema_decay=0.999,
        attack_steps=10, validation_attack_steps=20,
        max_validation_batches=8,
    )
    train_stage(
        DATA_PATH, FINAL_CHECKPOINT,
        init_checkpoint=BASE,
        objective='mixed', clean_weight=0.25,
        epochs=1, batch_size=128, learning_rate=0.005,
        warmup_epochs=1, ema_decay=0.99,
        attack_steps=10, validation_attack_steps=20,
        max_validation_batches=16,
    )
else:
    print('Skipping training; using attached final checkpoint:', FINAL_CHECKPOINT)


In [ ]:
verify_submission(FINAL_CHECKPOINT, model_name='resnet18')
evaluate_fgsm_pgd(FINAL_CHECKPOINT, DATA_PATH, model_name='resnet18')
print('Submit path:', FINAL_CHECKPOINT)
print('model_name: resnet18')
